# Audela Notebook Model Registration
This notebook is the guided experimentation layer for ML Studio.

Use it in this order:
1. Run Cell 3 to configure the target project and BI endpoints.
2. Run Cell 4 to load helper functions and builders.
3. Run Cell 5 to inspect the quick-start guidance.
4. Run Cell 6 to list BI datasets and preview SQL.
5. Run Cell 7 to build a model payload.
6. Run Cell 8 to use the guided toolbar and save to Audela.

Template version: nbtpl-2026.05.10-dev

## Detailed Tutorial
1. Prepare project workspace in ML Studio Notebooks page.
2. Open this notebook inside tenant project notebooks folder.
3. Use list_bi_datasets() and preview_bi_dataset(...) to work with BI sources.
4. Use MODEL_BUILDERS to produce valid model_data.
5. Run save_to_audela(payload) or click Save to Audela button.
6. Open Models page and continue Predict / Deploy flow.

In [ ]:
# 1) Configure Audela target
import os
import requests
from pprint import pprint

BASE_URL = "http://127.0.0.1:5000"
SOURCE_ID = 1  # replace with your BI source id
MODEL_NAME = "Notebook Linear Demo"
SQL_TEXT = "SELECT month_idx AS x, amount AS y FROM your_training_view"
X_COLUMN = "x"
Y_COLUMN = "y"
BI_SOURCES_URL = f"{BASE_URL.rstrip('/')}/ml/notebook/bi-sources"
BI_PREVIEW_URL = f"{BASE_URL.rstrip('/')}/ml/notebook/bi-preview"
BI_SCHEMA_URL = f"{BASE_URL.rstrip('/')}/ml/notebook/bi-schema"
REGISTER_URL = f"{BASE_URL.rstrip('/')}/ml/register-notebook-model"
metrics = {"r2": 0.81, "rmse": 3.2, "accuracy_type": "regression"}


In [ ]:
# 2) Predefined model builders (only supported algorithms)
def build_linear_regression(*, slope, intercept):
    return "linear_regression", {"algorithm": "linear_regression", "slope": float(slope), "intercept": float(intercept)}

def build_moving_average(*, tail_y=None, mean_y=None):
    if not tail_y and mean_y is None:
        raise ValueError("moving_average requires tail_y or mean_y")
    data = {"algorithm": "moving_average"}
    if tail_y:
        data["tail_y"] = [float(v) for v in tail_y]
    if mean_y is not None:
        data["mean_y"] = float(mean_y)
    return "moving_average", data

def build_mean_baseline(*, mean_y):
    return "mean_baseline", {"algorithm": "mean_baseline", "mean_y": float(mean_y)}

def build_decision_tree(*, threshold, left_mean, right_mean):
    return "decision_tree", {"algorithm": "decision_tree", "threshold": float(threshold), "left_mean": float(left_mean), "right_mean": float(right_mean)}

def build_random_forest(*, trees):
    if not isinstance(trees, list) or not trees:
        raise ValueError("random_forest requires a non-empty trees list")
    return "random_forest", {"algorithm": "random_forest", "trees": trees}

def build_kmeans_clustering(*, centroids, cluster_sizes=None, feature_names=None):
    if not isinstance(centroids, list) or not centroids:
        raise ValueError("kmeans_clustering requires centroids")
    out = {"algorithm": "kmeans_clustering", "centroids": centroids}
    if cluster_sizes is not None:
        out["cluster_sizes"] = cluster_sizes
    if feature_names is not None:
        out["feature_names"] = feature_names
    return "kmeans_clustering", out

MODEL_BUILDERS = {
    "linear_regression": build_linear_regression,
    "moving_average": build_moving_average,
    "mean_baseline": build_mean_baseline,
    "decision_tree": build_decision_tree,
    "random_forest": build_random_forest,
    "kmeans_clustering": build_kmeans_clustering,
}

def _auth_headers_and_session(session_cookie=None, auth_token=None):
    session = requests.Session()
    headers = {"Accept": "application/json"}

    cookie = (session_cookie or os.getenv("AUDELA_SESSION_COOKIE") or "").strip()
    token = (auth_token or os.getenv("AUDELA_AUTH_TOKEN") or "").strip()

    if cookie:
        session.cookies.set("session", cookie)
    if token:
        headers["Authorization"] = f"Bearer {token}"

    return session, headers

def list_bi_datasets(session_cookie=None, auth_token=None):
    session, headers = _auth_headers_and_session(session_cookie=session_cookie, auth_token=auth_token)
    response = session.get(BI_SOURCES_URL, headers=headers, timeout=60)
    data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {"ok": False, "raw": response.text}
    print("HTTP", response.status_code)
    pprint(data)
    return data

def preview_bi_dataset(*, source_id, sql_text, row_limit=50, session_cookie=None, auth_token=None):
    session, headers = _auth_headers_and_session(session_cookie=session_cookie, auth_token=auth_token)
    req = {"source_id": int(source_id), "sql_text": str(sql_text), "row_limit": int(row_limit)}
    response = session.post(BI_PREVIEW_URL, headers=headers, json=req, timeout=120)
    data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {"ok": False, "raw": response.text}
    print("HTTP", response.status_code)
    if data.get("ok"):
        print("Columns:", data.get("columns", []))
        print("Rows preview:", len(data.get("rows", [])))
    else:
        pprint(data)
    return data

def schema_bi_dataset(*, source_id, sql_text, row_limit=120, session_cookie=None, auth_token=None):
    session, headers = _auth_headers_and_session(session_cookie=session_cookie, auth_token=auth_token)
    req = {"source_id": int(source_id), "sql_text": str(sql_text), "row_limit": int(row_limit)}
    response = session.post(BI_SCHEMA_URL, headers=headers, json=req, timeout=120)
    data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {"ok": False, "raw": response.text}
    print("HTTP", response.status_code)
    pprint(data)
    return data

def save_to_audela(payload, base_url=BASE_URL, session_cookie=None, auth_token=None, timeout=60):
    session, headers = _auth_headers_and_session(session_cookie=session_cookie, auth_token=auth_token)
    url = REGISTER_URL if base_url == BASE_URL else f"{base_url.rstrip('/')}/ml/register-notebook-model"
    response = session.post(url, json=payload, headers=headers, timeout=timeout)

    try:
        data = response.json()
    except Exception:
        data = {"ok": False, "raw": response.text}

    if response.status_code in (401, 403):
        print("Authentication required in Audela. Add AUDELA_SESSION_COOKIE or pass session_cookie.")

    print("HTTP", response.status_code)
    print(data)
    return response, data


In [ ]:
# 3) Guided quick-start panel
def show_quick_start():
    print("AUDELA notebook quick start")
    print("Step 1: Run datasets = list_bi_datasets()")
    print("Step 2: Choose a source and test preview_bi_dataset(...)")
    print("Step 3: Build model_data with MODEL_BUILDERS[...]")
    print("Step 4: Update payload values")
    print("Step 5: Run save_to_audela(payload) or use the toolbar")
    print("Supported builders:", ", ".join(sorted(MODEL_BUILDERS.keys())))

show_quick_start()

In [ ]:
# 4) Build payload from predefined function + save
algorithm, model_data = MODEL_BUILDERS["linear_regression"](slope=1.25, intercept=12.0)

payload = {
    "model_name": MODEL_NAME,
    "algorithm": algorithm,
    "source_id": SOURCE_ID,
    "sql_text": SQL_TEXT,
    "x_column": X_COLUMN,
    "y_column": Y_COLUMN,
    "model_data": model_data,
    "metrics": metrics,
    "params": {"origin": "jupyter"}
}

# Optional if not already authenticated: set env AUDELA_SESSION_COOKIE in the notebook kernel.
resp, data = save_to_audela(payload)

In [ ]:
# 5) BI dataset integration examples
datasets = list_bi_datasets()
# preview = preview_bi_dataset(source_id=SOURCE_ID, sql_text=SQL_TEXT, row_limit=30)
# schema = schema_bi_dataset(source_id=SOURCE_ID, sql_text=SQL_TEXT, row_limit=120)

In [ ]:
# 6) Optional guided functions toolbar
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    project_sql = widgets.Textarea(value=SQL_TEXT, description="SQL", layout=widgets.Layout(width="100%", height="90px"))
    source_dropdown = widgets.Dropdown(options=[(f"Source #{SOURCE_ID}", SOURCE_ID)], description="Dataset")
    model_name_input = widgets.Text(value=MODEL_NAME, description="Model")
    schema_btn = widgets.Button(description="Schema profile", button_style="primary", icon="table")
    list_btn = widgets.Button(description="List BI datasets", button_style="info", icon="list")
    preview_btn = widgets.Button(description="Preview BI query", button_style="warning", icon="search")
    save_btn = widgets.Button(description="Save to Audela", button_style="success", icon="save")
    save_out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px"))
    help_html = widgets.HTML("<b>Guide:</b> 1) List datasets  2) Preview SQL  3) Save to Audela")

    def _refresh_sources():
        data = list_bi_datasets()
        items = data.get("items") if isinstance(data, dict) else []
        opts = [(f"{it.get('name', 'source')} (#{it.get('id', 0)})", int(it.get('id', 0))) for it in items if int(it.get('id', 0)) > 0]
        if opts:
            source_dropdown.options = opts
            source_dropdown.value = opts[0][1]

    def _on_list(_):
        with save_out:
            clear_output()
            _refresh_sources()

    def _on_preview(_):
        with save_out:
            clear_output()
            preview_bi_dataset(source_id=source_dropdown.value, sql_text=project_sql.value, row_limit=30)

    def _on_schema(_):
        with save_out:
            clear_output()
            schema_bi_dataset(source_id=source_dropdown.value, sql_text=project_sql.value, row_limit=120)

    def _on_save(_):
        with save_out:
            clear_output()
            payload["source_id"] = int(source_dropdown.value)
            payload["sql_text"] = str(project_sql.value)
            payload["model_name"] = str(model_name_input.value or payload.get("model_name") or "Notebook Model")
            save_to_audela(payload)

    schema_btn.on_click(_on_schema)
    list_btn.on_click(_on_list)
    preview_btn.on_click(_on_preview)
    save_btn.on_click(_on_save)
    display(help_html, model_name_input, source_dropdown, project_sql, widgets.HBox([list_btn, preview_btn, schema_btn, save_btn]), save_out)
    _refresh_sources()
except Exception:
    print("ipywidgets not available. Install with: pip install ipywidgets")
    print("Fallback helpers: list_bi_datasets(), preview_bi_dataset(...), schema_bi_dataset(...), save_to_audela(payload)")